# 🧪 Praktikum 08 – Agenten-Architekturen: ReAct & Reflection
**Applied Generative AI – NLP | Sommersemester 2026**

> 🎯 **Lernziele:** Einen autonomen Loop mit State-Management bauen · Komplexe Tool-Aufrufe mit Argumenten parsen · Self-Reflection zur Fehlerkorrektur einsetzen

⏱️ **Dauer:** 90 Minuten

In [ ]:
import os, sys, subprocess, shutil, time, requests, re
MODEL = "qwen3.5:0.8b"
def install_if_missing(pkgs):
    cmd = [sys.executable, "-m", "pip", "install"] + (["--break-system-packages"] if sys.prefix == getattr(sys, "base_prefix", sys.prefix) else [])
    subprocess.check_call(cmd + pkgs)
install_if_missing(["ollama", "requests"])
import ollama
print("✅ Setup abgeschlossen.")

## Teil 1 – Der Plan-and-Execute Agent ⏱️ 30 min
Statt direkt zu antworten, erstellt der Agent erst einen Plan.

In [ ]:
def get_plan(task):
    prompt = f"Erstelle einen 3-Schritte-Plan, um diese Aufgabe zu lösen: {task}\nAntworte nur als Liste."
    res = ollama.chat(model=MODEL, messages=[{"role": "user", "content": prompt}])
    return res['message']['content']

task = "Recherchiere das Wetter in Berlin, berechne die Kleidungsempfehlung und schreibe eine E-Mail."
print("PLAN:\n", get_plan(task))

## Teil 2 – ReAct Loop mit Memory ⏱️ 40 min
Wir bauen ein Gedächtnis ein, damit der Agent weiß, was er bereits getan hat.

In [ ]:
class SimpleAgent:
    def __init__(self, model):
        self.model = model
        self.memory = []
        self.tools = {"calc": lambda x: str(eval(x))}

    def run(self, query):
        self.memory.append({"role": "user", "content": f"Task: {query}\nFormat: THOUGHT, ACTION: tool(arg), PAUSE"})
        
        for _ in range(3):
            res = ollama.chat(model=self.model, messages=self.memory)['message']['content']
            print(f"Agent: {res}")
            self.memory.append({"role": "assistant", "content": res})
            
            match = re.search(r"ACTION: (\w+)\((.*?)\)", res)
            if match:
                tool, arg = match.groups()
                obs = self.tools.get(tool, lambda x: "Tool nicht gefunden")(arg)
                print(f"Tool Output: {obs}")
                self.memory.append({"role": "user", "content": f"OBSERVATION: {obs}"})
            else: break

agent = SimpleAgent(MODEL)
agent.run("Berechne 15 * 15 und dann plus 10.")

## Teil 3 – Reflection (Self-Criticism) ⏱️ 20 min
Der Agent soll seine eigene Antwort bewerten und ggf. korrigieren.

In [ ]:
def reflect_and_fix(answer):
    prompt = f"Kritisiere die folgende Antwort auf logische Fehler und korrigiere sie falls nötig: {answer}"
    res = ollama.chat(model=MODEL, messages=[{"role": "user", "content": prompt}])
    return res['message']['content']

wrong_ans = "Ein Kilo Federn ist schwerer als ein Kilo Blei, weil Federn mehr Volumen haben."
print("Korrektur:\n", reflect_and_fix(wrong_ans))

## 🧩 Aufgaben
1. **Argument Parsing:** Erweitere den Agenten so, dass er Tools mit zwei Argumenten (z.B. `add(a, b)`) versteht.
2. **State Export:** Speichere die `self.memory` Liste nach einem Run als JSON-Datei ab. Warum ist das für Debugging hilfreich?
3. **Stop-Kriterium:** Implementiere ein Limit für die Anzahl der Tool-Aufrufe, um Endlosschleifen zu verhindern.